In [1]:
import numpy as np

**Standard Inequality QP:**

$\underset{x}{\min} \quad \frac{1}{2} x^\top Q x + q^\top x$ 

$\text{s.t.} \quad Gx \leq h, \qquad Ax = b$

$ Q \in \mathbb{S}_+^n, \quad q \in \mathbb{R}^n$

$ G \in \mathbb{R}^{p \times n}, \quad h \in \mathbb{R}^p $

$ A \in \mathbb{R}^{m \times n}, \quad b \in \mathbb{R}^m $

**Solve by introducing a slack variable:**

$ \underset{x}{\min} \quad \frac{1}{2} x^\top Q x + q^\top x $ 

$ \text{s.t.} \quad Gx + s = h, \qquad Ax = b, \qquad s \geq 0 $

$ s \in \mathbb{R}^p $

**Dual variables:**

$ \text{Dual Equality Variables: } y \in \mathbb{R}^m $

$ \text{Dual Inequality Variables: } z \in \mathbb{R}^p $

**KKT Condtions:**

$ \text{s.t.} \quad Gx + s = h, \qquad Ax = b, \qquad s \geq 0 $

$ z \geq 0 $

$ Qx + q + G^\top z + A^\top y = 0 $

$ z_i s_i = 0, \quad i = 1, \ldots, p $

**Main KKT system to solve:**

$$
\underbrace{\begin{bmatrix}
Q & 0 & G^\top & A^T \\
0 & Z & S & 0 \\
G & I & 0 & 0 \\
A & 0 & 0 & 0
\end{bmatrix}}_{(n + 2p + m)\times(n + 2p + m)}
\underbrace{\begin{bmatrix}
\Delta x^{\text{aff}} \\
\Delta s^{\text{aff}} \\
\Delta z^{\text{aff}} \\
\Delta y^{\text{aff}}
\end{bmatrix}}_{(n + 2p + m)\times1}
=
\underbrace{\begin{bmatrix}
-(A^\top y + G^\top z + Q x + q) \\
-S z \\
-(G x + s - h) \\
-(A x - b)
\end{bmatrix}}_{(n + 2p + m)\times1}
$$

In [2]:
class Problem:
    """Convex Quadratic Program (QP) class to store problem parameters.

    min 1/2 x^T Q x + q^T x

    s.t. Gx <= h
         Ax = b

    Parameters
    ----------
    Q : (n, n) ndarray
        Quadratic cost matrix, symmetric positive semi-definite.
    q : (n,) ndarray
        Linear cost vector.
    G : (p, n) ndarray
        Linear inequality constraint matrix.
    h : (p,) ndarray
        Linear inequality constraint vector.
    A : (m, n) ndarray
        Linear equality constraint matrix.
    b : (m,) ndarray
        Linear equality constraint vector.

    """

    def __init__(self, Q: np.ndarray, q: np.ndarray, G: np.ndarray, h: np.ndarray, A: np.ndarray, b: np.ndarray) -> None:  # noqa: E501, PLR0913
        """Initialize the Problem class with input problem parameters."""
        self.Q = Q
        self.q = q
        self.G = G
        self.h = h
        self.A = A
        self.b = b

In [4]:
class Solver:
    """Primal-dual interior point method solver for convex QPs."""

    def __init__(self, problem: Problem) -> None:
        """Initialize the solver class with an input problem."""
        self.problem = problem
        self.n = np.size(problem.q)
        self.m = np.size(problem.b)
        self.p = np.size(problem.h)

    def initial_point(self) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Calculate the initial point (x0, s0, y0, z0)."""
        # Setup linear system
        LHS_row1 = np.block([self.problem.Q, self.problem.G.T, self.problem.A.T])
        LHS_row2 = np.block([self.problem.G, -np.eye(self.p), np.zeros([self.p, self.m])])
        LHS_row3 = np.block([self.problem.A, np.zeros([self.m, self.p]), np.zeros([self.m, self.m])])
        LHS = np.vstack([LHS_row1, LHS_row2, LHS_row3])

        RHS = np.vstack([-self.problem.q, self.problem.h, self.problem.b])

        # Solve linear system
        sol = np.linalg.solve(LHS, RHS)
        x = sol[:self.n]
        y = sol[-self.m:]

        # Set initial primal/dual variables
        x0 = x
        y0 = y

        z = self.problem.G @ x - self.problem.h

        alpha_p = np.max(z)

        s0 = -z if alpha_p < 0 else -z + 1 + alpha_p

        alpha_d = -np.min(z)

        z0 = z if alpha_d < 0 else z + 1 + alpha_d

        return x0, s0, z0, y0